# Exploring NHL API to get Data for Expected Goals Model

In [1]:
import pandas as pd
import requests
import json
from typing import List
import time
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

## Display

In [2]:
def adjust_df_display(dimension, action):
    """This function when called adjusts the output display of pandas dataframes. It either changes the max_columns or max_rows to infinite or resets those
    values to their default display limits.

    Args:
        dimension (string): Display dimension of a dataframe to alter; should be either "columns" or "rows".
        action (string): Action to be carried out on display settings; should be either "max" or "limit".
    """
    if dimension == "columns" and action == "max":
        pd.set_option('display.max_columns', None)
    elif dimension == "rows" and action == "max":
        pd.set_option('display.max_rows', None)
    elif dimension == "columns" and action == "limit":
        pd.reset_option('max_columns')
    else:
        pd.reset_option('max_rows')

In [3]:
adjust_df_display("columns", "max")
adjust_df_display("rows", "max")

## API Data

In [4]:
def normalize_coordinates(x: float | None,y: float | None,is_home_team: bool,home_team_defending_side: str | None) -> tuple[float | None, float | None, bool]:
    """
    Normalize shot coordinates so the shooting team always attacks
    in the positive x direction.

    Returns:
        (x_coord, y_coord, coord_normalized)
        coord_normalized is True only when a flip was actually applied.
        Fallback cases return raw coordinates with coord_normalized=False.
    """
    if x is None or y is None:
        return x, y, False

    if home_team_defending_side is None:
        # TODO: Implement fallback for seasons without 'homeTeamDefendingSide'
        return x, y, False

    if home_team_defending_side == "left":
        attacking_left = not is_home_team
    elif home_team_defending_side == "right":
        attacking_left = is_home_team
    else:
        # Unexpected value — return raw, flag as not normalized
        return x, y, False

    if attacking_left:
        return -x, y, True
    else:
        # Shooting team already attacks right — no flip needed, but
        # normalization logic ran successfully
        return x, y, True

In [ ]:
# Most up to date version
def extract_shot_events(data: dict, game_id: str) -> list[dict]:
    """
    Extracts relevant information from shot-related events (goal, shot-on-goal, missed-shot),
    including shooter team info and game context.

    Parameters:
    - data: full dictionary returned from the NHL play-by-play API for one game
    - game_id: string identifier for the game

    Returns:
    - List of dictionaries with structured shot event info.
    """
    shot_events = []
    valid_types = {"goal", "shot-on-goal", "missed-shot"}
    play_data = data.get("plays", [])

    # Team context
    home_team = data.get("homeTeam", {})
    away_team = data.get("awayTeam", {})
    
    created_at_utc = datetime.now(timezone.utc)
    created_at_et = created_at_utc.astimezone(ZoneInfo("America/Toronto"))

    for event in play_data:
        event_type = event.get("typeDescKey")
        if event_type not in valid_types:
            continue

        details = event.get("details", {})
        # Identify shooter ID and their team
        shooter_id = (
            details.get("scoringPlayerId") if event_type == "goal"
            else details.get("shootingPlayerId")
        )
        shooter_team_id = details.get("eventOwnerTeamId")

        # Determine if shooter is on home or away team
        is_home_team = shooter_team_id == home_team.get("id")
        shooter_team_abbrev = home_team.get("abbrev") if is_home_team else away_team.get("abbrev")
        opponent_team_abbrev = away_team.get("abbrev") if is_home_team else home_team.get("abbrev")
        
        # Parse situation code and compute strength state
        # 4-digit situation code has this format: A-G-S-H  → Away goalie, Away skaters, Home skaters, Home goalie
        situation_code = event.get("situationCode")
        if isinstance(situation_code, int):
            situation_code = str(situation_code)

        if not (isinstance(situation_code, str) and len(situation_code) == 4 and situation_code.isdigit()):
            strength_state = None
            away_goalie_pulled = None
            home_goalie_pulled = None
            shooting_team_strength_state = None
            shooting_team_strength_diff = None
        else:
            away_goalie_pulled = situation_code[0] == "0"
            home_goalie_pulled = situation_code[3] == "0"
            # strength_state is 'Away skaters'v'Home skaters' e.g. '4v5' for Home team on powerplay
            strength_state = f"{situation_code[1]}v{situation_code[2]}"
            
            # Derive skater counts
            away_skaters = int(situation_code[1])
            home_skaters = int(situation_code[2])

            if is_home_team is True:
                shooting_team_skaters = home_skaters
                defending_team_skaters = away_skaters
            elif is_home_team is False:
                shooting_team_skaters = away_skaters
                defending_team_skaters = home_skaters
            else:
                shooting_team_skaters = None
                defending_team_skaters = None

            if shooting_team_skaters is not None and defending_team_skaters is not None:
                shooting_team_strength_state = f"{shooting_team_skaters}v{defending_team_skaters}"
                shooting_team_strength_diff = shooting_team_skaters - defending_team_skaters
            else:
                shooting_team_strength_state = None
                shooting_team_strength_diff = None
                
        x_coord = details.get("xCoord")
        y_coord = details.get("yCoord")
        home_team_defending_side = event.get("homeTeamDefendingSide")
        x_norm, y_norm, coord_normalized = normalize_coordinates(x_coord, y_coord, is_home_team, home_team_defending_side)

        try:
            shot_info = {
                # Game context
                "game_id": game_id,
                "event_id": event.get("eventId"),
                "sort_order": event.get("sortOrder"),
                "period": event.get("periodDescriptor", {}).get("number"),
                "period_type": event.get("periodDescriptor", {}).get("periodType"),
                "time_in_period": event.get("timeInPeriod"),
                "time_remaining": event.get("timeRemaining"),
                "situation_code": situation_code,
                "strength_state": strength_state,
                "away_goalie_pulled": away_goalie_pulled,
                "home_goalie_pulled": home_goalie_pulled,
                "shooting_team_strength_state": shooting_team_strength_state,
                "shooting_team_strength_diff": shooting_team_strength_diff,

                # Player & team info
                "shooter_id": shooter_id,
                "goalie_id": details.get("goalieInNetId"),
                "shooter_team_id": shooter_team_id,
                "shooter_team_abbrev": shooter_team_abbrev,
                "opponent_team_abbrev": opponent_team_abbrev,
                "is_home_team": is_home_team,

                # Shot event info
                "event_type": event_type,
                "x_coord_raw": x_coord,
                "y_coord_raw": y_coord,
                "x_coord": x_norm,
                "y_coord": y_norm,
                "coord_normalized": coord_normalized,
                "zone": details.get("zoneCode"),
                "shot_type": details.get("shotType"),
                "is_goal": True if event_type == "goal" else False,
                
                # Miss reason
                "miss_reason": details.get("reason") if event_type == "missed-shot" else None,
                
                # Created at
                "created_at_utc": created_at_utc,
                "created_at_et": created_at_et,
            }

            shot_events.append(shot_info)

        except Exception as e:
            print(f"Skipping event due to error: {e}")
            continue

    return shot_events



In [6]:
# def get_shot_data_for_game(game_id: str) -> list[dict]:
#     try:
#         url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
#         response = requests.get(url)
#         response.raise_for_status()
#         data = response.json()
#         return extract_shot_events(data, game_id)
#     except Exception as e:
#         print(f"Failed for game {game_id}: {e}")
#         return []

In [12]:
import requests
import time
# New version to fix API call silently hanging
def get_shot_data_for_game(game_id: str) -> list[dict]:
    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"

    for attempt in range(3):  # retry up to 3 times
        try:
            response = requests.get(url, timeout=10)  # <-- CRITICAL FIX
            response.raise_for_status()
            data = response.json()
            return extract_shot_events(data, game_id)
        except requests.exceptions.Timeout:
            print(f"Timeout for game {game_id} (attempt {attempt+1}/3)")
            time.sleep(2 * (attempt + 1))  # exponential backoff
        except Exception as e:
            print(f"Failed for game {game_id}: {e}")
            return []

    print(f"Failed after retries for game {game_id}")
    return []

In [21]:
def get_nhl_team_abbreviations() -> list[str]:
    url = "https://api-web.nhle.com/v1/standings/now"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    abbrevs = []
    for team_record in data.get("standings", []):
        team_abbrev_info = team_record.get("teamAbbrev", {})
        abbrev = team_abbrev_info.get("default")
        if abbrev:
            abbrevs.append(abbrev)

    return sorted(set(abbrevs))

# Example usage
team_abbrevs = get_nhl_team_abbreviations()

In [15]:
def get_all_regular_season_game_ids(season: str, team_abbrevs: list[str]):
    game_ids = set()
    for team in team_abbrevs:
        url = f"https://api-web.nhle.com/v1/club-schedule-season/{team}/{season}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        
        for g in data.get("games", []):
            if g.get("gameType") == 2:
                game_ids.add(g["id"])
    return sorted(game_ids)

# Example usage
season = '20232024'
game_ids = get_all_regular_season_game_ids(season, team_abbrevs)
print(f"Found {len(game_ids)} regular-season games for {season}")

Found 1312 regular-season games for 20232024


In [ ]:
for season in ['20192020', '20202021', '20212022', '20222023', '20232024', '20242025']:
    game_ids = get_all_regular_season_game_ids(season, team_abbrevs)
    print(f"{season} season: {len(game_ids)} regular season games")

In [25]:
# Took 50 mins to run last time
all_shots = []

seasons = ['20192020', '20202021', '20212022', '20222023', '20232024', '20242025']

for season in seasons:
    print(f"Starting season {season}...")
    game_ids = get_all_regular_season_game_ids(season, team_abbrevs)
    for i, game_id in enumerate(game_ids):
        shots = get_shot_data_for_game(game_id)
        if shots:
            all_shots.extend(shots)
        if (i + 1) % 200 == 0:
            print(f"  Processed {i + 1} games from season {season}")
        time.sleep(0.1)  # Gentle pacing to avoid hammering server

# Convert to DataFrame and save
df = pd.DataFrame(all_shots)
#df.to_csv("nhl_shots_2019_2024.csv", index=False)
print("Finished collecting shot data.")
df.shape

Starting season 20192020...
  Processed 200 games from season 20192020
Timeout for game 2019020387 (attempt 1/3)
  Processed 400 games from season 20192020
  Processed 600 games from season 20192020
  Processed 800 games from season 20192020
  Processed 1000 games from season 20192020
Starting season 20202021...
  Processed 200 games from season 20202021
  Processed 400 games from season 20202021
  Processed 600 games from season 20202021
  Processed 800 games from season 20202021
Starting season 20212022...
  Processed 200 games from season 20212022
Timeout for game 2021020313 (attempt 1/3)
  Processed 400 games from season 20212022
Timeout for game 2021020491 (attempt 1/3)
  Processed 600 games from season 20212022
  Processed 800 games from season 20212022
  Processed 1000 games from season 20212022
  Processed 1200 games from season 20212022
Starting season 20222023...
  Processed 200 games from season 20222023
  Processed 400 games from season 20222023
  Processed 600 games from s

(622481, 32)

In [26]:
df.to_csv("nhl_shots_2019_2024.csv", index=False)

In [ ]:
df.shape

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

In [ ]:
game_ids = ["2024020954", "2024020955", "2024020956", "2024020957", "2024020958", "2024020959", "2024020960", "2024020961", "2024020962", "2024020963"]

In [ ]:
all_shots = []

for game_id in game_ids:
    shots = get_shot_data_for_game(game_id)
    all_shots.extend(shots)

df = pd.DataFrame(all_shots)
df.head(20)

In [ ]:
def audit_event_fields(game_id: str):
    data = get_shot_data_for_game(game_id)
    if not data:
        print(f"No data found for {game_id}")
        return

    sample_event = data[0]
    print(f"Fields in event {sample_event['event_id']} from {game_id}:")
    for key in sample_event.keys():
        print(f"- {key}")

In [ ]:
def audit_event_fields(game_id: str):
    try:
        url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        return f"Failed to retrieve data for game {game_id}: {e}"

    play_data = data.get("plays", [])
    valid_types = {"shot-on-goal", "missed-shot"}

    for event in play_data:
        if event.get("typeDescKey") in valid_types:
            details = event.get("details", {})
            return {
                "game_id": game_id,
                "event_id": event.get("eventId"),
                "top_level_keys": list(event.keys()),
                "details_keys": list(details.keys()) if isinstance(details, dict) else "No details"
            }

    return f"No valid shot events found in game {game_id}"

game_ids = [
    "2015020100", "2016020100", "2017020100", "2018020100", "2019020100", "2020020100",
    "2021020100", "2022020100", "2023020100", "2024020100"
]

for gid in game_ids:
    result = audit_event_fields(gid)
    print(result)

In [ ]:
def print_first_shot_event(game_id: str):
    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
    response = requests.get(url)
    data = response.json()

    print(f"\n==== Game ID: {game_id} ====")
    plays = data.get("plays", [])
    for play in plays:
        if play.get("typeDescKey") in {"goal", "shot-on-goal", "missed-shot"}:
            print("Full event:")
            print(json.dumps(play, indent=2))
            print("\nDetails field only:")
            print(json.dumps(play.get("details", {}), indent=2))
            return
    print("No shot events found.")

# Sample Game IDs from 2017–18 and 2018–19 seasons
game_ids = [
    "2017020100", "2018020100"  # 2018–19 season
]

for gid in game_ids:
    print_first_shot_event(gid)

In [13]:
all_shots = []
game_ids = ["2024020954"]
for i, game_id in enumerate(game_ids):
    shots = get_shot_data_for_game(game_id)
if shots:
    all_shots.extend(shots)
df = pd.DataFrame(all_shots)
#df.to_csv("nhl_shots_2019_2024.csv", index=False)
df.head(10)

,game_id,event_id,sort_order,period,period_type,time_in_period,time_remaining,situation_code,strength_state,away_goalie_pulled,home_goalie_pulled,shooting_team_strength_state,shooting_team_strength_diff,score_state,score_differential,shooter_id,goalie_id,shooter_team_id,shooter_team_abbrev,opponent_team_abbrev,is_home_team,event_type,x_coord_raw,y_coord_raw,x_coord,y_coord,coord_normalized,zone,shot_type,is_goal,miss_reason,created_at_utc,created_at_et
0,2024020954,103,12,1,REG,00:09,19:51,1551,5v5,False,False,5v5,0,None,None,8477505,8476999.0,28,SJS,OTT,False,shot-on-goal,-55,1,55,1,True,O,wrist,False,None,2026-03-26 17:06:02.803899+00:00,2026-03-26 13:06:02.803899-04:00
1,2024020954,115,26,1,REG,01:09,18:51,1551,5v5,False,False,5v5,0,None,None,8480801,8477970.0,9,OTT,SJS,True,missed-shot,41,37,41,37,True,O,wrist,False,high-and-wide-left,2026-03-26 17:06:02.803899+00:00,2026-03-26 13:06:02.803899-04:00
2,2024020954,116,28,1,REG,01:12,18:48,1551,5v5,False,False,5v5,0,None,None,8482245,8477970.0,9,OTT,SJS,True,missed-shot,54,-39,54,-39,True,O,wrist,False,wide-left,2026-03-26 17:06:02.803899+00:00,2026-03-26 13:06:02.803899-04:00
3,2024020954,131,40,1,REG,02:11,17:49,1551,5v5,False,False,5v5,0,None,None,8480848,8476999.0,28,SJS,OTT,False,shot-on-goal,-56,11,56,11,True,O,wrist,False,None,2026-03-26 17:06:02.803899+00:00,2026-03-26 13:06:02.803899-04:00
4,2024020954,140,52,1,REG,02:57,17:03,1551,5v5,False,False,5v5,0,None,None,8484911,8476999.0,28,SJS,OTT,False,shot-on-goal,-62,-15,62,-15,True,O,wrist,False,None,2026-03-26 17:06:02.803899+00:00,2026-03-26 13:06:02.803899-04:00
5,2024020954,148,61,1,REG,03:34,16:26,1551,5v5,False,False,5v5,0,None,None,8480801,8477970.0,9,OTT,SJS,True,shot-on-goal,51,-18,51,-18,True,O,wrist,False,None,2026-03-26 17:06:02.803899+00:00,2026-03-26 13:06:02.803899-04:00
6,2024020954,155,67,1,REG,03:59,16:01,1551,5v5,False,False,5v5,0,None,None,8482144,8476999.0,28,SJS,OTT,False,shot-on-goal,-54,32,54,32,True,O,wrist,False,None,2026-03-26 17:06:02.803899+00:00,2026-03-26 13:06:02.803899-04:00
7,2024020954,162,73,1,REG,04:29,15:31,1551,5v5,False,False,5v5,0,None,None,8482144,8476999.0,28,SJS,OTT,False,shot-on-goal,46,39,-46,39,True,D,wrist,False,None,2026-03-26 17:06:02.803899+00:00,2026-03-26 13:06:02.803899-04:00
8,2024020954,186,98,1,REG,06:06,13:54,1551,5v5,False,False,5v5,0,None,None,8477505,8476999.0,28,SJS,OTT,False,missed-shot,-50,-2,50,-2,True,O,wrist,False,wide-right,2026-03-26 17:06:02.803899+00:00,2026-03-26 13:06:02.803899-04:00
9,2024020954,199,112,1,REG,07:16,12:44,1551,5v5,False,False,5v5,0,None,None,8478013,8476999.0,28,SJS,OTT,False,shot-on-goal,-36,25,36,25,True,O,slap,False,None,2026-03-26 17:06:02.803899+00:00,2026-03-26 13:06:02.803899-04:00


In [14]:
df.dtypes

game_id                                                  object
event_id                                                  int64
sort_order                                                int64
period                                                    int64
period_type                                              object
time_in_period                                           object
time_remaining                                           object
situation_code                                           object
strength_state                                           object
away_goalie_pulled                                         bool
home_goalie_pulled                                         bool
shooting_team_strength_state                             object
shooting_team_strength_diff                               int64
score_state                                              object
score_differential                                       object
shooter_id                              